In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA causal_project;

In [0]:
%sql
SELECT
  treatment,
  COUNT(*)                        AS n,
  AVG(pre_avg_weekly_spend)       AS avg_weekly_spend,
  AVG(pre_coupon_usage_rate)      AS coupon_rate,
  AVG(pre_loyalty_card_rate)      AS loyalty_rate,
  AVG(pre_avg_weekly_trips)       AS weekly_trips,
  AVG(pre_department_breadth)     AS dept_breadth,
  AVG(avg_daily_campaign_spend)   AS outcome,
  AVG(clean_window_days) AS campaign_days,
  AVG(pre_active_weeks) AS pre_active_weeks
FROM causal_project.gold_household
GROUP BY treatment;

In [0]:
gold_df = spark.table("gold_household").toPandas()
gold_df.head()

In [0]:
import numpy as np
import pandas as pd

confounders = [
    'pre_avg_weekly_spend',
    'pre_avg_weekly_trips', 
    'pre_coupon_usage_rate',
    'pre_loyalty_card_rate',
    'pre_department_breadth',
    'pre_active_weeks',
    'clean_window_days'
]

treated   = gold_df[gold_df['treatment'] == 1]
untreated = gold_df[gold_df['treatment'] == 0]

smd_rows = []
for col in confounders:
    mean_t  = treated[col].mean()
    mean_c  = untreated[col].mean()
    std_pool = np.sqrt(
        (treated[col].var() + untreated[col].var()) / 2
    )
    smd = (mean_t - mean_c) / std_pool
    smd_rows.append({
        'feature':    col,
        'mean_typeA': round(mean_t, 3),
        'mean_typeBC': round(mean_c, 3),
        'SMD':        round(smd, 3)
    })

smd_df = pd.DataFrame(smd_rows).sort_values('SMD')
smd_df

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['pre_avg_weekly_spend', 'pre_loyalty_card_rate']):
    treated[col].hist(
        bins=40, alpha=0.6, label='TypeA (personalized)', 
        ax=ax, density=True
    )
    untreated[col].hist(
        bins=40, alpha=0.6, label='TypeB/C (non-personalized)', 
        ax=ax, density=True
    )
    ax.set_title(col)
    ax.legend()

plt.suptitle('Pre-campaign feature distributions by treatment group')
plt.tight_layout()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gold_df['avg_daily_campaign_spend'].hist(bins=50, ax=axes[0])
axes[0].set_title('Outcome — raw')
axes[0].set_xlabel('avg_daily_campaign_spend')

np.log1p(gold_df['avg_daily_campaign_spend']).hist(bins=50, ax=axes[1])
axes[1].set_title('Outcome — log1p transformed')
axes[1].set_xlabel('log1p(avg_daily_campaign_spend)')

skew_raw = gold_df['avg_daily_campaign_spend'].skew()
skew_log = np.log1p(gold_df['avg_daily_campaign_spend']).skew()

axes[0].set_xlabel(f'skewness: {skew_raw:.2f}')
axes[1].set_xlabel(f'skewness: {skew_log:.2f}')

plt.tight_layout()

## EDA Summary & Modeling Decisions

### Treatment Balance (Standardized Mean Differences)

The SMD table confirms meaningful pre-campaign differences between 
TypeA (personalized) and TypeB/C (non-personalized) households across 
all behavioral features. All SMDs exceed the 0.1 threshold conventionally 
used to indicate meaningful imbalance in observational studies.

| Feature | TypeA mean | TypeB/C mean | SMD |
|---|---|---|---|
| pre_avg_weekly_spend | 71.10 | 84.61 | -0.315 |
| pre_department_breadth | 12.85 | 13.52 | -0.233 |
| pre_avg_weekly_trips | 2.21 | 2.56 | -0.229 |
| pre_active_weeks | 32.26 | 34.54 | -0.207 |
| pre_loyalty_card_rate | 11.66 | 12.95 | -0.199 |
| pre_coupon_usage_rate | 0.34 | 0.42 | -0.099 |
| clean_window_days | 43.95 | 30.24 | 1.109 |

**Interpretation**: TypeA households were lighter shoppers across every 
behavioral dimension before the campaign began: lower spend, fewer trips, 
narrower category breadth, lower coupon and loyalty card usage. This is 
consistent with the retailer using personalized campaigns as a targeted 
intervention to lift engagement among less active households.

The large SMD on `clean_window_days` (1.109) reflects the truncation 
applied to 176 households with overlapping campaigns (TypeA campaign 
windows were on average 14 days longer than TypeB/C windows). This 
variable is included as a technical control in the causal model.

**This imbalance confirms that naive comparison of campaign-period spend 
between TypeA and TypeB/C households would be confounded by pre-existing 
behavioral differences. Causal inference methods are necessary to isolate 
the true treatment effect.**

---

### Outcome Distribution

Raw `avg_daily_campaign_spend` shows right skewness of 1.52, driven by 
a small number of high-spending households. log1p transformation reduces 
skewness to -0.44, producing an approximately symmetric distribution 
suitable for DML estimation.

**Decision: log1p(avg_daily_campaign_spend) is used as the outcome 
variable in all causal models.** ATEs estimated on the log scale are 
interpretable as approximate percentage effects on daily spend.

The 14 zero-outcome households (households that received a campaign but 
made no purchases during the campaign window) map to log1p(0) = 0, 
preserving them in the analysis without requiring special treatment.

---